# Make fight music with AI

Makes three 30-second fight tracks with **MusicGen** (Meta's free music model), each shaped so it loops without a jump.
Listen to them at the bottom, then send the one you like (or all three) back for the scroll site.

**How to use**
1. Menu **Runtime → Change runtime type → T4 GPU**.
2. Menu **Runtime → Run all**. It takes about 10 minutes, most of it the first download of the model.
3. A zip called `fight_music.zip` downloads at the end.

To try different music, change the descriptions in Step 2 and run Steps 2 to 4 again.
MusicGen's model is free for non-commercial use, which fits a personal project.

### Step 1: load the music model (about 5 minutes the first time)

In [ ]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration

MODEL = "facebook/musicgen-large"   # best quality; "facebook/musicgen-medium" is faster and lighter
gpu = torch.cuda.is_available()
processor = AutoProcessor.from_pretrained(MODEL)
model = MusicgenForConditionalGeneration.from_pretrained(MODEL, dtype=torch.float16 if gpu else torch.float32)
model = model.to("cuda" if gpu else "cpu").eval()
RATE = model.config.audio_encoder.sampling_rate
print("Loaded on the", "graphics card" if gpu else "processor (very slow: choose the T4 GPU runtime)")

### Step 2: describe the music

In [ ]:
# One track per description. Name: what the music should sound like.
TRACKS = {
    "orchestral": "epic orchestral anime battle music, fast taiko drums and pounding toms, aggressive staccato strings, "
                  "brass stabs, rising tension, sword duel, 150 bpm, instrumental",
    "dark_fantasy": "intense dark fantasy fight theme, driving cello ostinato, war drums, low choir, heroic horn melody, "
                    "cinematic, 140 bpm, instrumental",
    "hybrid": "hybrid trailer battle music, pulsing synth bass with orchestral percussion, big drum hits, "
              "tense and energetic, 145 bpm, instrumental",
}
SECONDS = 30   # MusicGen makes up to about 30 seconds at a time

### Step 3: make the tracks (about a minute each)

In [ ]:
import numpy as np, soundfile as sf, subprocess, os
from IPython.display import Audio, display

os.makedirs("fight_music", exist_ok=True)
for n, (name, description) in enumerate(TRACKS.items()):
    torch.manual_seed(n)
    inputs = processor(text=[description], padding=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        audio = model.generate(**inputs, do_sample=True, guidance_scale=3.0, max_new_tokens=int(SECONDS * 50))
    x = audio[0, 0].float().cpu().numpy()

    # make it loop: blend the last 1.5 seconds into the start, so the end runs straight back into the beginning
    fade = int(1.5 * RATE)
    ramp = np.linspace(0, 1, fade, dtype=np.float32)
    loop = x[:-fade].copy()
    loop[:fade] = x[:fade] * ramp + x[-fade:] * (1 - ramp)

    wav = f"fight_music/{name}.wav"
    sf.write(wav, loop, RATE)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", wav, "-af", "loudnorm=I=-18:TP=-1.5",
                    "-ar", "44100", "-c:a", "libmp3lame", "-b:a", "128k", wav[:-4] + ".mp3"], check=True)
    os.remove(wav)
    print(f"{name}: {len(loop) / RATE:.0f} s")
    display(Audio(wav[:-4] + ".mp3"))

### Step 4: download

In [ ]:
from google.colab import files
!cd fight_music && zip -qr ../fight_music.zip .
files.download("fight_music.zip")